In [0]:

from pyspark.sql import SparkSession
from pyspark.sql import functions as F

spark = SparkSession.builder.appName("dataframe").getOrCreate()

In [0]:
data =[(1,"Amit",50000),
       (2,"Rahul",60000),
       (3,"Priya",70000),
       (1,"Ajay",50000)]
df = spark.createDataFrame(data,["id","name","salary"])
display(df)

In [0]:
df.groupBy("id").agg(F.count("id").alias("count")).filter(F.col("count")>1).show()

In [0]:
# repartition() vs coalesce()

# repartition(n):
#   - Does a FULL SHUFFLE of the data across the cluster
#   - Can be used to INCREASE or DECREASE the number of partitions
#   - Produces evenly-sized partitions (balanced data distribution)
#   - Expensive: data is moved between executors over the network

# coalesce(n):
#   - Avoids a full shuffle; merges existing partitions on the same executor
#   - Can only DECREASE the number of partitions (asking for more is a no-op)
#   - Cheaper than repartition, but partitions may be uneven in size
#   - Typical use: reduce partitions before writing output or after filtering

# Demo with the df from the earlier cell
print("Current partitions:", spark.sparkContext.defaultParallelism)

df_repartitioned = df.repartition(8)  # full shuffle -> 8 even partitions
df_coalesced = df.coalesce(2)         # no shuffle, just merges existing partitions
df_repartitioned_before = df.repartition(8).coalesce(2)  # no shuffle: reuse the 8 partitions

df_repartitioned.groupBy(F.spark_partition_id().alias("partition")).count().show()
df_coalesced.groupBy(F.spark_partition_id().alias("partition")).count().show()

# Rule of thumb:
# - Going from fewer -> more partitions, or you need evenly balanced partitions -> repartition()
# - Shrinking partitions cheaply (e.g., before write, after a big filter) -> coalesce()

In [0]:
display(df)

In [0]:
df2=spark.sql("select * from employee")

In [0]:
display(df2)

In [0]:
df2.select("department","salary").groupBy("department").agg(F.avg("salary").alias("avg_salary")).show()

In [0]:
# Optimizing a join between two large DataFrames (millions of rows each)
#
# 1) Start with broadcast join for small(er) dimension tables
#    If one side fits in executor memory (default ~10MB, adjustable via
#    spark.sql.autoBroadcastJoinThreshold), broadcast it so every executor has
#    a local copy -> NO shuffle of the big table at all.

from pyspark.sql import functions as F

# Explicit broadcast (use when the small side is *slightly* above the auto threshold
# but still safely fits in memory):
df_big = df2                                   # millions of rows (fact table)
df_small = df.groupBy("id").agg(F.avg("salary").alias("avg_salary"))  # small dimension

joined = df_big.join(F.broadcast(df_small), df_big["id"] == df_small["id"], "left")
joined.explain(True)   # look for BroadcastHashJoin (no Exchange = no shuffle)

# 2) If BOTH sides are large -> shuffle hash / sort-merge join. Optimize the shuffle:
#    - Pre-partition both sides on the join key with repartition(), and reuse
#      the same DataFrames for multiple joins so Spark skips re-shuffling:
#      df_a = df_a.repartition(200, "id")
#      df_b = df_b.repartition(200, "id")
#      df_a.join(df_b, "id")   # Spark sees both are already hash-partitioned on id
#
#    - Right-size the number of shuffle partitions to your data volume
#      (rule of thumb ~128MB per partition):
#      spark.conf.set("spark.sql.shuffle.partitions", 400)   # default is 200
#
# 3) Skew handling: if a few join keys dominate (e.g. NULLs or one hot key),
#    Spark 3+ AQE fixes most of it automatically -> keep these on:
#      spark.conf.set("spark.sql.adaptive.enabled", "true")
#      spark.conf.set("spark.sql.adaptive.skewJoin.enabled", "true")
#      spark.conf.set("spark.sql.adaptive.coalescePartitions.enabled", "true")
#    Manual fallback: split hot keys into a separate broadcast-able bucket and
#    union the results back.
#
# 4) Reduce data BEFORE the join (biggest win, often forgotten):
#    - Filter each side first, push predicates down
#    - Select only the columns you need (drops shuffle volume)
#    - Deduplicate keys if the join would explode the row count
#
#      df_a = df_a.select("id", "col_needed").filter(F.col("ts") > cutoff).repartition("id")
#
# 5) Storage-level optimizations for repeated joins:
#    - Delta tables: cluster (Liquid Clustering) or Z-ORDER by the join key so
#      file pruning reduces rows read per join
#    - cache()/persist() a heavily reused intermediate only if it's reused 3+ times
#
# Decision cheat-sheet:
#   one side < a few GB, definitely < 8GB        -> broadcast join
#   both large, evenly distributed keys         -> shuffle join + AQE, tune shuffle.partitions
#   both large, skewed keys                    -> AQE skew join, or salting
#   joining on the same key repeatedly          -> pre-partition / cluster by that key